# 02 · Gemma — iteración manual del pipeline de texto

En el benchmark inicial se compararon tres familias de modelos abiertas sobre los mismos chunks del corpus: **Qwen3-4B-Instruct-2507, Gemma 3 4B IT y Mistral 7B Instruct v0.3**.

Para continuar el prototipado se selecciona **Gemma 3 4B IT** porque, en la muestra inicial:

- produjo salida JSON válida de forma consistente;
- mostró buena adherencia al texto fuente;
- tuvo una latencia competitiva;
- con 4B parámetros resulta razonable para iterar en Google Colab.

Esta decisión es **pragmática y provisional**: el objetivo de este notebook no es establecer un benchmark exhaustivo, sino depurar manualmente el comportamiento del pipeline antes de automatizarlo.

## Dos estrategias que vamos a comparar

### Pipeline 1 — Extract → Select → Write
`chunk → hasta 3 ideas → selección de la idea relevante → texto final corto`

Esta estrategia fuerza al modelo a separar extracción factual y redacción.

### Pipeline 2 — Direct summary
`chunk completo → texto final de ~30 segundos`

Esta estrategia sirve como baseline simple para comprobar si la descomposición del Pipeline 1 aporta valor.

In [ ]:
!pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes huggingface_hub pandas

## 1. Clonar FinTok y cargar el corpus

In [ ]:
from pathlib import Path
import json, random, gc, re, time
import numpy as np
import pandas as pd
import torch

# Clonar solo si todavía no existe en esta sesión de Colab
if not Path("/content/FinTok").exists():
    !git clone https://github.com/mirdbg/FinTok.git

REPO_DIR = Path("/content/FinTok")
ZIP_PATH = REPO_DIR / "data" / "corpus_miax_2026.zip"
WORK_DIR = Path("/content/corpus_miax_2026")

WORK_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f"No encuentro: {ZIP_PATH}"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Corpus:", ZIP_PATH)

In [ ]:
import zipfile

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(WORK_DIR)

with open(WORK_DIR / "chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(chunks)

print(f"Chunks cargados: {len(df):,}")
display(df[["chunk_id", "ticker", "fiscal_year", "item", "n_tokens", "contiene_tabla"]].head())

## 2. Cargar Gemma

El notebook usa cuantización 4-bit para reducir VRAM.

`HF_TOKEN` debe estar guardado en **Secrets** de Colab y tener activado *Acceso desde el cuaderno*.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)
print("Hugging Face autenticado.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-4b-it"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    quantization_config=quant_config,
)

model.eval()
print("Modelo cargado:", MODEL_ID)

## 3. Función común de generación

In [ ]:
def ask_gemma(system_prompt, user_prompt, max_new_tokens=600):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.perf_counter() - start
    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(generated, skip_special_tokens=True).strip(), elapsed

## 4. Elegir manualmente un chunk

Durante esta fase queremos **iterar manualmente**, así que no hacemos todavía un bucle por los 1.749 chunks.

Puedes filtrar el DataFrame, leer candidatos y cambiar `CHUNK_ID` tantas veces como quieras.

In [ ]:
# Algunos candidatos para explorar
display(
    df.sample(15, random_state=42)[
        ["chunk_id", "ticker", "fiscal_year", "item", "n_tokens", "contiene_tabla"]
    ]
)

In [ ]:
# CAMBIA ESTE ID para probar otro chunk
CHUNK_ID = df.sample(1, random_state=42).iloc[0]["chunk_id"]

row = df.loc[df["chunk_id"] == CHUNK_ID].iloc[0]
chunk_text = row["texto"]

print("CHUNK:", row["chunk_id"])
print("TICKER:", row["ticker"])
print("AÑO:", row["fiscal_year"])
print("ITEM:", row["item"])
print("=" * 100)
print(chunk_text)

# PIPELINE 1 · Extract → Select → Write

Esta celda realiza deliberadamente **tres operaciones dentro de una única llamada**:

1. Extrae hasta tres ideas factuales del chunk.
2. Decide si alguna tiene suficiente valor para FinTok.
3. Si existe una idea adecuada, redacta un texto corto y autosuficiente.

El modelo puede devolver `DISCARD`. No queremos obligarlo a fabricar contenido a partir de un fragmento irrelevante.

Durante la iteración manual pedimos también que muestre las ideas candidatas y la razón de selección. Más adelante podremos ocultar esa traza y conservar únicamente `final_text`.

In [ ]:
PIPELINE_1_SYSTEM = """
You are the editorial engine of FinTok, a system that transforms official
10-K filings into short, accurate and interesting financial content.

You must work ONLY with the information contained in the supplied excerpt.
Do not use external knowledge. Do not invent numbers, causes or conclusions.

Perform these steps internally and return the requested structured output:

STEP 1 — EXTRACT
Extract up to 3 distinct, self-contained factual ideas from the excerpt.
Return fewer than 3 if fewer meaningful ideas exist.

Prefer:
- important quantitative results or changes,
- year-over-year developments,
- business drivers,
- strategic developments,
- customer/supplier dependencies,
- material risks,
- facts that are understandable without reading the entire filing.

STEP 2 — SELECT
Decide whether any extracted idea is genuinely worth turning into short
financial content for a general but financially interested audience.

Be selective. A fact can be correct but still be too technical, routine,
administrative or context-dependent to make good content.

Select at most ONE idea.

STEP 3 — WRITE
If an idea is selected, rewrite it as a concise, professional piece of
financial content.

The final text must:
- communicate one clear idea;
- remain faithful to the filing;
- include the relevant company context;
- be understandable without seeing the original chunk;
- sound natural when read aloud;
- avoid clickbait;
- avoid investment advice;
- avoid adding interpretation not supported by the source;
- be approximately 1–3 short sentences.

Return ONLY valid JSON using this schema:

{
  "candidate_ideas": [
    {
      "idea": "...",
      "evidence": "..."
    }
  ],
  "decision": "KEEP or DISCARD",
  "selected_idea": "..." or null,
  "reason": "...",
  "final_text": "..." or null
}
""".strip()

pipeline_1_user = f"""
COMPANY/TICKER: {row['ticker']}
FISCAL YEAR: {row['fiscal_year']}
10-K ITEM: {row['item']}
CHUNK ID: {row['chunk_id']}

10-K EXCERPT:
----------------
{chunk_text}
----------------
""".strip()

output_1, latency_1 = ask_gemma(
    PIPELINE_1_SYSTEM,
    pipeline_1_user,
    max_new_tokens=650
)

print(f"Latencia: {latency_1:.2f}s")
print(output_1)

# PIPELINE 2 · Direct 30-second summary

Aquí Gemma recibe directamente el chunk completo.

No le mostramos las ideas ni la selección del Pipeline 1. Queremos un **baseline independiente**:

> “De este fragmento, ¿qué contarías en aproximadamente 30 segundos?”

Como referencia, un texto hablado de 30 segundos suele estar aproximadamente en el entorno de **60–80 palabras**, dependiendo del ritmo. Usamos ese rango como guía, no como restricción rígida.

In [ ]:
PIPELINE_2_SYSTEM = """
You are the editorial engine of FinTok, a system that transforms official
10-K filings into short, accurate and understandable financial content.

Read the supplied 10-K excerpt and create a short spoken summary containing
the most worthwhile information to communicate to a general but financially
interested audience.

The summary should:
- take approximately 30 seconds to read aloud;
- target roughly 60–80 words;
- focus on the most relevant information rather than covering everything;
- be self-contained;
- clearly identify the company when necessary;
- preserve important figures and context;
- use natural spoken language;
- remain professional and informative;
- avoid clickbait;
- avoid investment advice;
- use ONLY information supported by the supplied excerpt;
- never invent or infer unsupported information.

If the excerpt is highly technical, routine or weak as standalone content,
still summarize its most meaningful information faithfully rather than
artificially making it exciting.

Return ONLY the final spoken text. Do not explain your reasoning.
""".strip()

pipeline_2_user = f"""
COMPANY/TICKER: {row['ticker']}
FISCAL YEAR: {row['fiscal_year']}
10-K ITEM: {row['item']}
CHUNK ID: {row['chunk_id']}

10-K EXCERPT:
----------------
{chunk_text}
----------------
""".strip()

output_2, latency_2 = ask_gemma(
    PIPELINE_2_SYSTEM,
    pipeline_2_user,
    max_new_tokens=220
)

print(f"Latencia: {latency_2:.2f}s")
print()
print(output_2)
print()
print("Palabras:", len(output_2.split()))

## 5. Comparación manual

Por ahora no automatizamos una métrica. Lee ambas salidas y pregúntate:

- ¿Cuál identifica mejor lo importante?
- ¿Hay alguna afirmación no soportada por el chunk?
- ¿El Pipeline 1 descarta demasiado o demasiado poco?
- ¿La selección de una sola idea funciona?
- ¿El Pipeline 2 intenta meter demasiadas cosas?
- ¿Suena natural leído en voz alta?
- ¿30 segundos es una duración adecuada?
- ¿Qué prompt cambiarías?

La intención de este notebook es precisamente **iterar sobre estas decisiones antes de congelar el pipeline**.

In [ ]:
print("=" * 100)
print("ORIGINAL")
print("=" * 100)
print(chunk_text)

print("\n\n" + "=" * 100)
print("PIPELINE 1 — EXTRACT → SELECT → WRITE")
print("=" * 100)
print(output_1)

print("\n\n" + "=" * 100)
print("PIPELINE 2 — DIRECT 30s SUMMARY")
print("=" * 100)
print(output_2)

## 6. Guardar una iteración manual

Cuando encuentres un ejemplo interesante, puedes ejecutar esta celda para ir construyendo un pequeño registro de pruebas.

In [ ]:
manual_result = {
    "chunk_id": row["chunk_id"],
    "ticker": row["ticker"],
    "fiscal_year": int(row["fiscal_year"]),
    "item": str(row["item"]),
    "original_text": chunk_text,
    "pipeline_1_output": output_1,
    "pipeline_1_latency": latency_1,
    "pipeline_2_output": output_2,
    "pipeline_2_latency": latency_2,
}

print(json.dumps(manual_result, indent=2, ensure_ascii=False))

## Próximos pasos

Cuando el comportamiento sea satisfactorio, este notebook se puede convertir en un pipeline automático:

`todos los chunks → Pipeline 1 → KEEP → generación del guion definitivo → voz / imagen / vídeo`

Antes de automatizarlo conviene decidir manualmente:
1. qué entendemos por una idea suficientemente interesante;
2. cuánto debe durar el contenido;
3. qué tono editorial queremos;
4. si una sola idea por vídeo es la estrategia adecuada.